# Debug: Why does run_ensemble_cancer_risk.py produce 0 rows?

Systematically check each early-return condition in `run_one`.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/orcd/pool/003/dbertsim_shared/ukb/cancer_risk"
LABELS_PATH = "/orcd/pool/003/dbertsim_shared/ukb"

MODELS_COX_XGB = ["cox", "xgb", "rsf"]
MODELS_PROB_NAMING = ["ordinal"]
MODELS = MODELS_COX_XGB + MODELS_PROB_NAMING

TIME_TO_RISK_COL = {1: "risk_1years", 2: "risk_2years", 5: "risk_5years", 10: "risk_10years"}
TIME_TO_PROB_COL = {1: "prob_1yr", 2: "prob_2yr", 5: "prob_5yr", 10: "prob_10yr"}

FEMALE_SPECIFIC_CANCERS = {"breast_cancer", "ovarian_cancer", "uterine_cancer"}
MALE_SPECIFIC_CANCERS = {"prostate_cancer"}

CANCER_TYPES = [
    "breast_cancer", "colorectal_cancer", "kidney_cancer", "lung_cancer",
    "lymphoma_cancer", "melanoma_cancer", "ovarian_cancer", "prostate_cancer",
    "skin_cancer", "uterine_cancer",
]

TIME_FRAME = 5

## 1. Check that the label CSVs exist and load

In [2]:
label_files = [
    "ukb_cancer_train.csv", "ukb_cancer_valid.csv", "ukb_cancer_test.csv",
    "ukb_cancer_train_female.csv", "ukb_cancer_valid_female.csv", "ukb_cancer_test_female.csv",
    "ukb_cancer_train_male.csv", "ukb_cancer_valid_male.csv", "ukb_cancer_test_male.csv",
]
for f in label_files:
    path = os.path.join(LABELS_PATH, f)
    exists = os.path.isfile(path)
    print(f"  {f}: exists={exists}")

  ukb_cancer_train.csv: exists=True
  ukb_cancer_valid.csv: exists=True
  ukb_cancer_test.csv: exists=True
  ukb_cancer_train_female.csv: exists=True
  ukb_cancer_valid_female.csv: exists=True
  ukb_cancer_test_female.csv: exists=True
  ukb_cancer_train_male.csv: exists=True
  ukb_cancer_valid_male.csv: exists=True
  ukb_cancer_test_male.csv: exists=True


In [3]:
valid_df = pd.read_csv(os.path.join(LABELS_PATH, "ukb_cancer_valid.csv"), low_memory=False)
test_df = pd.read_csv(os.path.join(LABELS_PATH, "ukb_cancer_test.csv"), low_memory=False)
valid_f = pd.read_csv(os.path.join(LABELS_PATH, "ukb_cancer_valid_female.csv"), low_memory=False)
test_f = pd.read_csv(os.path.join(LABELS_PATH, "ukb_cancer_test_female.csv"), low_memory=False)
valid_m = pd.read_csv(os.path.join(LABELS_PATH, "ukb_cancer_valid_male.csv"), low_memory=False)
test_m = pd.read_csv(os.path.join(LABELS_PATH, "ukb_cancer_test_male.csv"), low_memory=False)

cancer_sets = {
    "": (valid_df, test_df),
    "_female": (valid_f, test_f),
    "_male": (valid_m, test_m),
}
print(f"Default valid: {valid_df.shape}, test: {test_df.shape}")
print(f"Female  valid: {valid_f.shape}, test: {test_f.shape}")
print(f"Male    valid: {valid_m.shape}, test: {test_m.shape}")

Default valid: (10519, 3015), test: (10520, 3015)
Female  valid: (5713, 3004), test: (5714, 3004)
Male    valid: (4886, 3000), test: (4886, 3000)


## 2. Check label columns exist for each cancer in its cohort

In [4]:
def get_suffix(diag_type):
    if diag_type in FEMALE_SPECIFIC_CANCERS:
        return "_female"
    if diag_type in MALE_SPECIFIC_CANCERS:
        return "_male"
    return ""

for cancer in CANCER_TYPES:
    suffix = get_suffix(cancer)
    vdf, tdf = cancer_sets[suffix]
    has_label = cancer in vdf.columns
    time_col = f"{cancer}_time_to_diagnosis"
    has_time = time_col in vdf.columns
    print(f"  {cancer:30s} (cohort: {suffix or 'default':8s}) | label_col={has_label} | time_col={has_time}")

  breast_cancer                  (cohort: _female ) | label_col=True | time_col=True
  colorectal_cancer              (cohort: default ) | label_col=True | time_col=True
  kidney_cancer                  (cohort: default ) | label_col=True | time_col=True
  lung_cancer                    (cohort: default ) | label_col=True | time_col=True
  lymphoma_cancer                (cohort: default ) | label_col=True | time_col=True
  melanoma_cancer                (cohort: default ) | label_col=True | time_col=True
  ovarian_cancer                 (cohort: _female ) | label_col=True | time_col=True
  prostate_cancer                (cohort: _male   ) | label_col=True | time_col=True
  skin_cancer                    (cohort: default ) | label_col=True | time_col=True
  uterine_cancer                 (cohort: _female ) | label_col=True | time_col=True


## 3. Check get_labels_future (first early return in run_one)

If this returns `None`, `run_one` returns `[]` immediately.

In [6]:
def filter_for_future_prediction(df, diag_type):
    if diag_type not in df.columns:
        return df
    return df.loc[df[diag_type] == 0].copy()

def get_labels_future(df, diag_type, horizon_years):
    time_col = f"{diag_type}_time_to_diagnosis"
    if time_col not in df.columns:
        return None
    y = ((df[diag_type] == 0) & (df[time_col] <= horizon_years)).astype(int)
    return y

for cancer in CANCER_TYPES:
    suffix = get_suffix(cancer)
    vdf, tdf = cancer_sets[suffix]
    vf = filter_for_future_prediction(vdf, cancer)
    tf = filter_for_future_prediction(tdf, cancer)
    y_valid = get_labels_future(vf, cancer, TIME_FRAME)
    y_test = get_labels_future(tf, cancer, TIME_FRAME)
    v_status = f"len={len(y_valid)}, pos={y_valid.sum()}" if y_valid is not None else "None"
    t_status = f"len={len(y_test)}, pos={y_test.sum()}" if y_test is not None else "None"
    print(f"  {cancer:30s} | y_valid: {v_status:25s} | y_test: {t_status}")

  breast_cancer                  | y_valid: len=5539, pos=82          | y_test: len=5540, pos=82
  colorectal_cancer              | y_valid: len=10471, pos=47         | y_test: len=10471, pos=48
  kidney_cancer                  | y_valid: len=10508, pos=13         | y_test: len=10508, pos=14
  lung_cancer                    | y_valid: len=10510, pos=37         | y_test: len=10511, pos=37
  lymphoma_cancer                | y_valid: len=10491, pos=22         | y_test: len=10492, pos=23
  melanoma_cancer                | y_valid: len=10480, pos=25         | y_test: len=10480, pos=25
  ovarian_cancer                 | y_valid: len=5700, pos=12          | y_test: len=5701, pos=12
  prostate_cancer                | y_valid: len=4817, pos=91          | y_test: len=4816, pos=90
  skin_cancer                    | y_valid: len=10303, pos=183        | y_test: len=10304, pos=183
  uterine_cancer                 | y_valid: len=5690, pos=13          | y_test: len=5690, pos=13


## 4. Check risk CSV files exist (second early return)

`load_probs_for_split` returns `None` if **any** model's risk file is missing or lacks the required column.

In [7]:
print(f"DATA_PATH = {DATA_PATH}")
print(f"Contents of DATA_PATH: {os.listdir(DATA_PATH) if os.path.isdir(DATA_PATH) else 'DOES NOT EXIST'}")
print()

DATA_PATH = /orcd/pool/003/dbertsim_shared/ukb/cancer_risk
Contents of DATA_PATH: ['cox', 'rsf', 'xgb_probs', 'ordinal', 'xgb']



In [8]:
for model in MODELS:
    model_dir = os.path.join(DATA_PATH, model)
    exists = os.path.isdir(model_dir)
    if exists:
        files = os.listdir(model_dir)
        print(f"  {model}/ : {len(files)} files")
        for f in sorted(files)[:10]:
            print(f"    {f}")
        if len(files) > 10:
            print(f"    ... and {len(files) - 10} more")
    else:
        print(f"  {model}/ : DOES NOT EXIST  <--- this would cause load_probs_for_split to return None")

  cox/ : 30 files
    breast_cancer_cox_risk_test.csv
    breast_cancer_cox_risk_train.csv
    breast_cancer_cox_risk_valid.csv
    colorectal_cancer_cox_risk_test.csv
    colorectal_cancer_cox_risk_train.csv
    colorectal_cancer_cox_risk_valid.csv
    kidney_cancer_cox_risk_test.csv
    kidney_cancer_cox_risk_train.csv
    kidney_cancer_cox_risk_valid.csv
    lung_cancer_cox_risk_test.csv
    ... and 20 more
  xgb/ : 30 files
    breast_cancer_xgb_risk_test.csv
    breast_cancer_xgb_risk_train.csv
    breast_cancer_xgb_risk_valid.csv
    colorectal_cancer_xgb_risk_test.csv
    colorectal_cancer_xgb_risk_train.csv
    colorectal_cancer_xgb_risk_valid.csv
    kidney_cancer_xgb_risk_test.csv
    kidney_cancer_xgb_risk_train.csv
    kidney_cancer_xgb_risk_valid.csv
    lung_cancer_xgb_risk_test.csv
    ... and 20 more
  rsf/ : 20 files
    breast_cancer_demo_protein_blood_rsf_risk_test.csv
    breast_cancer_demo_protein_blood_rsf_risk_valid.csv
    colorectal_cancer_demo_protein_blood_rs

In [10]:
risk_col = TIME_TO_RISK_COL[TIME_FRAME]
prob_col = TIME_TO_PROB_COL[TIME_FRAME]

print(f"Looking for risk_col='{risk_col}' (cox/xgb/rsf) and prob_col='{prob_col}' (ordinal)")
print(f"Time frame = {TIME_FRAME} years\n")

for cancer in CANCER_TYPES:
    print(f"--- {cancer} ---")
    for dataset in ["valid", "test"]:
        for model in MODELS:
            is_cox_xgb = model in MODELS_COX_XGB
            if is_cox_xgb:
                fname = f"{cancer}_{model}_risk_{dataset}.csv"
            else:
                fname = f"{cancer}_{dataset}.csv"
            path = os.path.join(DATA_PATH, model, fname)
            exists = os.path.isfile(path)
            col = risk_col if is_cox_xgb else prob_col
            if exists:
                df = pd.read_csv(path, nrows=3, low_memory=False)
                has_col = col in df.columns
                has_eid = "eid" in df.columns
                status = f"OK (cols: eid={has_eid}, {col}={has_col})" if has_col and has_eid else f"MISSING COL (has: {list(df.columns[:8])})"
            else:
                status = "FILE NOT FOUND"
            print(f"  [{dataset:5s}] {model:8s} -> {fname:55s} | {status}")
    print()

Looking for risk_col='risk_5years' (cox/xgb/rsf) and prob_col='prob_5yr' (ordinal)
Time frame = 5 years

--- breast_cancer ---
  [valid] cox      -> breast_cancer_cox_risk_valid.csv                        | OK (cols: eid=True, risk_5years=True)
  [valid] xgb      -> breast_cancer_xgb_risk_valid.csv                        | OK (cols: eid=True, risk_5years=True)
  [valid] rsf      -> breast_cancer_rsf_risk_valid.csv                        | OK (cols: eid=True, risk_5years=True)
  [valid] ordinal  -> breast_cancer_valid.csv                                 | OK (cols: eid=True, prob_5yr=True)
  [test ] cox      -> breast_cancer_cox_risk_test.csv                         | OK (cols: eid=True, risk_5years=True)
  [test ] xgb      -> breast_cancer_xgb_risk_test.csv                         | OK (cols: eid=True, risk_5years=True)
  [test ] rsf      -> breast_cancer_rsf_risk_test.csv                         | OK (cols: eid=True, risk_5years=True)
  [test ] ordinal  -> breast_cancer_test.csv      

## 5. Try loading probs for one cancer to trace the exact failure

In [11]:
test_cancer = CANCER_TYPES[0]  # breast_cancer
print(f"Tracing load_probs_for_split for: {test_cancer} @ {TIME_FRAME}yr\n")

for dataset in ["valid", "test"]:
    print(f"  === {dataset} ===")
    for model in MODELS:
        is_cox_xgb = model in MODELS_COX_XGB
        if is_cox_xgb:
            fname = f"{test_cancer}_{model}_risk_{dataset}.csv"
        else:
            fname = f"{test_cancer}_{dataset}.csv"
        path = os.path.join(DATA_PATH, model, fname)
        if not os.path.isfile(path):
            print(f"    {model}: FILE MISSING -> {path}")
            print(f"    >>> load_probs_for_split would return None here <<<")
            continue
        df = pd.read_csv(path, low_memory=False).dropna(subset=["eid"])
        col = risk_col if is_cox_xgb else prob_col
        if col not in df.columns:
            print(f"    {model}: COLUMN '{col}' MISSING (has: {list(df.columns)})")
            print(f"    >>> load_probs_for_split would return None here <<<")
            continue
        print(f"    {model}: OK ({len(df)} rows, eid range [{df['eid'].min():.0f}, {df['eid'].max():.0f}])")

Tracing load_probs_for_split for: breast_cancer @ 5yr

  === valid ===
    cox: OK (10345 rows, eid range [1000029, 6020293])
    xgb: OK (5713 rows, eid range [1001980, 6021309])
    rsf: OK (5539 rows, eid range [1001980, 6021309])
    ordinal: OK (5713 rows, eid range [1001980, 6021309])
  === test ===
    cox: OK (5540 rows, eid range [1000029, 6020846])
    xgb: OK (5714 rows, eid range [1000029, 6020846])
    rsf: OK (5540 rows, eid range [1000029, 6020846])
    ordinal: OK (5714 rows, eid range [1000029, 6020846])


## 6. Summary: which early-return is being hit?

In [12]:
print("Cancer                          | Labels OK? | Valid probs? | Test probs?  | Would produce rows?")
print("-" * 100)

def load_probs_for_split(data_path, disease, time_frame, dataset):
    rc = TIME_TO_RISK_COL[time_frame]
    pc = TIME_TO_PROB_COL[time_frame]
    dfs = []
    for model in MODELS:
        is_cox_xgb = model in MODELS_COX_XGB
        if is_cox_xgb:
            fname = f"{disease}_{model}_risk_{dataset}.csv"
        else:
            fname = f"{disease}_{dataset}.csv"
        path = os.path.join(data_path, model, fname)
        if not os.path.isfile(path):
            return None, f"{model}: file missing ({fname})"
        df = pd.read_csv(path, low_memory=False).dropna(subset=["eid"])
        col = rc if is_cox_xgb else pc
        if col not in df.columns:
            return None, f"{model}: col '{col}' missing (has {list(df.columns)[:5]}...)"
        dfs.append(df[["eid", col]].rename(columns={col: model}))
    out = dfs[0]
    for d in dfs[1:]:
        out = out.merge(d, on="eid", how="inner")
    return out, "OK"

for cancer in CANCER_TYPES:
    suffix = get_suffix(cancer)
    vdf, tdf = cancer_sets[suffix]
    vf = filter_for_future_prediction(vdf, cancer)
    tf = filter_for_future_prediction(tdf, cancer)
    y_v = get_labels_future(vf, cancer, TIME_FRAME)
    y_t = get_labels_future(tf, cancer, TIME_FRAME)
    labels_ok = y_v is not None and y_t is not None

    vp, v_reason = load_probs_for_split(DATA_PATH, cancer, TIME_FRAME, "valid")
    tp, t_reason = load_probs_for_split(DATA_PATH, cancer, TIME_FRAME, "test")

    would_work = labels_ok and vp is not None and tp is not None
    label_str = "YES" if labels_ok else "NO"
    vp_str = f"YES ({len(vp)})" if vp is not None else f"NO: {v_reason}"
    tp_str = f"YES ({len(tp)})" if tp is not None else f"NO: {t_reason}"
    work_str = "YES" if would_work else "NO  <---"
    print(f"{cancer:30s} | {label_str:10s} | {vp_str:40s} | {tp_str:40s} | {work_str}")

Cancer                          | Labels OK? | Valid probs? | Test probs?  | Would produce rows?
----------------------------------------------------------------------------------------------------
breast_cancer                  | YES        | YES (1091)                               | YES (5540)                               | YES
colorectal_cancer              | YES        | YES (10471)                              | YES (10471)                              | YES
kidney_cancer                  | YES        | YES (10508)                              | YES (10508)                              | YES
lung_cancer                    | YES        | YES (10510)                              | YES (10511)                              | YES
lymphoma_cancer                | YES        | YES (10491)                              | YES (10492)                              | YES
melanoma_cancer                | YES        | YES (10480)                              | YES (10480)                      